# Introspection Factorization — gate runner (A100 80 GB)

Runs the whole chain under Garcia's **`workspace_band`** injection policy: **G0**
assets, **G1** lens validation, **G2** single-layer harness invariants, **G2b**
the band and both vector arms, the concept **re-pilot**, the **sweep** with
**G4a** integrity, the **cascade per arm** with **G4**, the figures, and **G5**.
Each gate emits a structured report and stops. Nothing prints PASS or FAIL — you
read the numbers and decide whether the next stage proceeds.

**Read this before running anything.** Two earlier runs bracketed the operating
regime without ever testing it.

1. Run 1 injected raw concept vectors, ‖v‖ ≈ 11, so realised ‖δ‖ was **11–88× the
   residual norm** — overwrite, not steering.
2. Run 2 unit-normalised the direction and then injected Garcia's *per-layer*
   strengths at a *single* layer. His `intervention_layer_policy: workspace_band`
   registers one intervention at **every layer 24–40** — seventeen of them, each
   scaled by the live median residual norm at that layer. One layer at the same
   per-layer α is ~17× weaker in cumulative displacement (**23×** measured on a
   toy stack, above 17× because live norms compound as the injections stack).
   Hence pilot detection ~1e-3 across all 315 concepts, G3 detection 0.000 in
   every cell, zero-tier collapse, AUC ≈ 0.5.

Every number from those two runs describes a misconfiguration. This notebook
runs the regime between them, which is also the regime that makes the
order-effect contrast against Garcia's 322 → 0 result apples-to-apples.

**§8 (G2b) is a decision point, not a checkpoint.** It reports dose-response and
coherence across the band for both arms; a *human* picks the operating strength
from it and writes it into `configs/sprint.yaml`. `src/config.py` refuses to
guess, so every stage after it stops with a message until that is done.

**Two arms, one grid.** `concept` = activation("Tell me about X") − baseline
mean, extracted at every band layer. `jlens_row` = (W_U[t] @ J_l) in layer-ℓ
space — the exact object Garcia injects, and the positive control: if it does
not reproduce his steering and report rates, the harness is at fault rather than
the model. At matched ‖δ‖ the pair is a causal manipulation of J-space loading.

**What to expect on a cold A100 80 GB**

| step | time | GPU | note |
|---|---|---|---|
| install | ~2 min | | |
| model download | 20–40 min | | 55.6 GB, **first run only** |
| G0 | 15–25 min | yes | dominated by 63 eigendecompositions of a 5120x5120 matrix; `--layers-stride 4` cuts it to ~6 min |
| G1 | ~25 min | yes | 80 items x 63 layers x 3 lenses |
| G2 | ~15 min | yes | single-layer invariants; superseded as a *measurement*, kept as a harness check |
| **G2b** | **~30 min** | yes | the band, both arms, the ladder. **Read it and choose the operating point.** |
| `01_concept_vectors` | ~25 min | yes | band re-pilot over the full pool, then per-layer extraction for both arms |
| G3 | ~30 min | yes | detection grid + controls + generation |
| `02_sweep` | ~15 min | yes | 1,920 cells; 17 hooks per forward, so a band prefill costs more than a single-layer one |
| `03_generate` | ~30 min | yes | 1,920 generations, operating point only |
| G4a | seconds | **no** | audits artifacts |
| `04_factors` x2 | ~20 min | yes | once per arm |
| G4 x2, figures x2, G5 x2 | seconds | **no** | one cascade per arm |

Every stage is a separate process and loads the model itself (~3 min from cache).
That is deliberate: every artifact is a file on disk, so any stage can be re-run
alone after a disconnect. `01_concept_vectors`, `02_sweep` and `03_generate` are
additionally resumable *within* themselves. **G4a, G4, the figures and G5 need no
GPU**, so you can re-run them any time to see what landed.

**Before you start:** the reports are the deliverable. They are written to
`artifacts/*/*_report.txt` as well as printed, so a disconnect does not lose them.

## 1 — Measure the GPU you actually have

Do not proceed on an assumption. Colab's High-RAM setting increases *system* RAM, not VRAM.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import subprocess
out = subprocess.run(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip().splitlines()
vram_gb = int(out[0]) / 1024 if out and out[0].strip().isdigit() else 0
print(f"\nVRAM detected: {vram_gb:.1f} GiB")
print("Tier A (target) needs ~62 GiB: 55.6 weights + 6.6 lens fp32 + activations.")
if vram_gb < 70:
    print("!! Below the Tier A assumption. G0 will still run and report actual")
    print("!! headroom, but G1/G2 at batch 8 may OOM. Lower --batch, or subsample")
    print("!! layers with --layers-stride.")

## 2 — Install

`jlens` is pinned to the commit the sprint was verified against. `diptest` is the one dependency beyond the build spec's list — see `ASSETS.md` for why.

In [ ]:
%pip -q install "transformers>=4.57.1" accelerate huggingface_hub numpy pandas scipy scikit-learn matplotlib pyyaml diptest
%pip -q install anthropic   # only used if ANTHROPIC_API_KEY is set (LLM judge)
%pip -q install "git+https://github.com/anthropics/jacobian-lens@581d398613e5602a5af361e1c34d3a92ea82ba8e"

import torch, transformers, jlens, diptest
print("torch       ", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("jlens       ", jlens.__file__)

## 3 — Point at the repo

Upload the `introspection-factorization` folder to the runtime (or `git clone` it)
and set `REPO` below. The cell verifies every file the gates need.

In [ ]:
import os, sys, json
from pathlib import Path

REPO = Path("/content/introspection-factorization")   # <-- edit if yours differs
if not REPO.exists():
    for candidate in (Path.cwd(), Path.cwd().parent,
                      Path("/content/drive/MyDrive/introspection-factorization")):
        if (candidate / "gates" / "g0_assets.py").exists():
            REPO = candidate
            break

required = [
    "configs/sprint.yaml", "configs/baseline_words.json",
    "configs/concepts.json", "configs/judge_rubrics.json",
    "configs/prereg.json",
    "src/stats.py", "src/lens.py", "src/inject.py", "src/band_inject.py",
    "src/config.py", "src/vectors.py",
    "src/prompts.py", "src/judge.py", "src/sweep.py", "src/factors.py",
    "src/plots.py",
    "gates/g0_assets.py", "gates/g1_lens.py", "gates/g2_inject.py",
    "gates/g2b_band.py", "gates/g3_baseline.py", "gates/g4a_sweep.py",
    "gates/g4_factors.py", "gates/g5_controls.py",
    "scripts/00_verify_dip.py", "scripts/00_verify_positions.py",
    "scripts/01_concept_vectors.py",
    "scripts/02_sweep.py", "scripts/03_generate.py",
    "scripts/04_factors.py", "scripts/05_figures.py",
]
missing = [f for f in required if not (REPO / f).exists()]
print("REPO =", REPO)
if missing:
    raise SystemExit(f"missing files: {missing}\nSet REPO to the repo root.")
print("all", len(required), "required files present")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
(REPO / "artifacts").mkdir(exist_ok=True)

pool = json.load(open("configs/concepts.json"))
print(f"concept pool: {pool['n']} words, source categories validated = {pool['category_counts_validated']}")
rub = json.load(open("configs/judge_rubrics.json"))
print(f"judge rubrics: {len(rub['criteria'])} vendored verbatim from eval_utils.py")

# the policy every stage below inherits, printed once so a stale config is
# visible here rather than three hours in
import config as cfg_mod
cfg = cfg_mod.load(REPO)
band = cfg_mod.injection_layers(cfg)
print()
print(f"injection policy : {cfg_mod.injection_policy(cfg)}")
print(f"band             : {len(band)} layers {band[0]}..{band[-1]}")
print(f"norm_mode        : {cfg_mod.norm_mode(cfg)}   (live = Garcia's protocol)")
print(f"arms             : {cfg_mod.vector_arms(cfg)}")
print(f"strengths (each layer): {cfg_mod.strengths(cfg)}"
      f"  extension {cfg['planned'].get('strengths_extension')}")
print(f"probe layer (f1) : {cfg_mod.probe_layer(cfg)}   "
      f"f2 primary readout: {cfg['planned']['f2_primary_layer']}")
op = cfg["planned"].get("operating_strength")
print(f"operating_strength: {op}"
      + ("   <- still unchosen; G2b (section 8) is where you pick it" if op is None else ""))

## 4 — Preflight: the statistics and the plumbing, on CPU

`src/stats.py` underpins every gate, so check it before spending an hour of GPU
time. This reproduces the dip comparison recorded in `ASSETS.md`: a hand-rolled
dip (wrong — it decouples the two segments at the mode), an independent linear
program, and the reference `diptest`. LP and diptest must agree; the hand-rolled
one will not.

`00_verify_positions.py` walks every prompt builder and the new task assignment:
`positions` must be a `list[int]` in bounds for all 10 tasks x 2 orders, and the
task groups must partition the concept list without exceeding the batch — a
group that mixed tasks could not share one forward, which is the invariant the
whole sweep's cost model rests on.

In [ ]:
!python scripts/00_verify_dip.py

In [ ]:
!python scripts/00_verify_positions.py

In [ ]:
import numpy as np, stats
rng = np.random.default_rng(0)
print("wilson(50,100)        ", stats.wilson(50,100))
print("wilson(0,10)          ", stats.wilson(0,10), " <- stays in [0,1]")
print("participation_ratio   ", round(stats.participation_ratio(np.ones(8)),3), "(flat 8-spectrum -> 8)")
print("dip, uniform n=60     ", {k:(round(v,4) if isinstance(v,float) else v) for k,v in stats.dip_test(rng.random(60), n_boot=2000).items()})
print("dip, bimodal n=60     ", {k:(round(v,4) if isinstance(v,float) else v) for k,v in stats.dip_test(np.r_[rng.normal(0,1,30),rng.normal(8,1,30)], n_boot=2000).items()})
print("MDE base .30 n=60     ", round(stats.min_detectable_effect(0.30,60),4), " <- weak power at concept granularity")
print("MDE base .30 n=4320   ", round(stats.min_detectable_effect(0.30,4320),4))

## 5 — GATE G0: assets and environment

First run downloads the model (55.6 GB) and the lens (3.3 GB). Both are cached
afterwards.

**Three numbers to read.** `lens fitted layers` — file-size arithmetic predicts 63
of 64; this reads `source_layers` directly, and if it disagrees the memory budget
is wrong. The **identity-cosine profile** — `W_U J_l` approaches `W_U` as `l`
approaches the final layer, so the cosine should climb toward ~1.0; flat or
non-monotonic means the layer indexing is wrong, which would invalidate the layer
band and therefore the choice of 27/31/35. And **headroom**, which decides the
batch size G1/G2 can afford.

**Runtime.** Budget 15–25 min after the model is cached, plus a one-time
20–40 min download on the first run. Nearly all of it is the effective-rank
cross-check: one `eigvalsh` on a 5120x5120 matrix per fitted layer, 63 of them.
For a fast first pass use `--layers-stride 4` — 16 layers instead of 63 still
shows the identity-cosine profile clearly and finishes in about 6 minutes. Note
the stride does *not* shorten the NaN/Inf/row-norm scan, which always covers
all 63 layers on CPU.


In [ ]:
!python gates/g0_assets.py 2>&1 | tee artifacts/g0_console.log

## 6 — GATE G1: lens validation

The most consequential gate. A lens that loads but does not work produces a
plausible f2 that is noise, and nothing downstream notices.

**Three numbers to read.** **Null-lens pass@1** — the random-rotation lens is the
placebo; if it scores near the Jacobian lens, the readout is coming through `W_U`
alone and the lens contributes nothing. **Median rank by layer** — where Garcia's
argument lives, and what a binary hit rate hides. **Retention counts** — if the
competence filter guts multihop, the depth-ordering cross-check loses its n.

Expect multilingual to retain poorly: its filter requires the model to emit e.g.
`pequeño` within 8 greedy tokens.

In [ ]:
!python gates/g1_lens.py --per-category 20 2>&1 | tee artifacts/g1_console.log

## 7 — GATE G2: single-layer harness invariants

**This gate no longer measures the experiment.** It injects at one layer, which
is the configuration that produced 1e-4 dose-response and 0.000 detection. It is
kept because its invariants are still the cheapest check that the hook machinery
is sound — and because the band's own invariants in G2b are the same properties
at 17 layers, so a disagreement between the two localises the fault.

**What must be exact.** `zero-strength identity` = 0.0 (adding `0*v` is exact in
floating point, so anything else means the hook does something besides add).
`spatial containment` outside the window = 0.0 at the injection layer **and every
layer below it**. Hook fire count = 1 in prefill and 1 across `generate` — that is
the check that nothing is injected during cached decode.

**What will not be exact.** `batch equivalence` — bf16 reductions are not
order-invariant, so B=4 batched vs 4 singles differs. The raw magnitude is
reported; you judge it.

**What is a finding, not an invariant.** Leakage to positions *after* the window
at layers *above* the injection layer. 48 of 64 blocks are linear-attention with
a causal conv (kernel 4), so information moves forward in position by
construction.

Skip this cell if you are short of time; G2b covers the same invariants under the
policy that actually runs.

In [ ]:
!python gates/g2_inject.py --n-concepts 10 --batch 8 2>&1 | tee artifacts/g2_console.log

## 8 — GATE G2b: the band, both arms, and the operating point

**The 30 minutes that stand between two invalid ladders and the sweep.** Nothing
downstream is meaningful until this report is read by a person.

**The number the whole change turns on** is under PRIMARY: cumulative
displacement of the 17-layer band versus a single layer at the *same* per-layer
α. A ratio near 1 would mean the band buys nothing and the diagnosis was wrong.
The toy-stack estimate was ~23×.

**The invariants**, all at 17 layers now: `zero-strength identity` = 0.0;
`layers fired` = 17 in prefill and 17 across `generate` with max 1 fire each;
containment outside the window = 0.0 *below* the band's first layer; and the
realised ‖δ_ℓ‖ / base_ℓ table, which the hook's contract fixes at α_rel exactly
at every layer — a layer that departs is a mis-scaled layer, named.

**The decision.** Read the dose-response table (probability *and* rank — at 1e-5
the probability is floor noise and rank is the informative quantity) against the
coherence table (NLL on a neutral task, scored by the unhooked model, so rising
NLL is damage and not successful steering). Pick the lowest α at which the target
rank is small and NLL has not moved.

**The stop condition, from the change order:** if arm `jlens_row` at α ≈
0.05–0.09 does not visibly steer the answers, **stop and debug against Garcia's
repo before burning the sweep.** That arm is his own object; if it does nothing,
the fault is here, not in the model.

**Arm B construction check** is under CROSS-CHECK: the cosine between the
analytic row `J_ℓᵀ(g ⊙ W_U[t])` and the same direction obtained by
differentiating the readout logit. 1.0 means the arm injects the direction the
lens reads. The `unfolded` column drops the final norm's gain, which is what a
reading of the lens that ignores the RMSNorm would produce — reported so the
choice is visible rather than assumed.

In [ ]:
!python gates/g2b_band.py --n-concepts 10 --batch 8 2>&1 | tee artifacts/g2b_console.log

### 8b — Choose the operating point, then record it

`planned.operating_strength` is `null` in `configs/sprint.yaml` and every stage
below reads it from there. This is deliberate: a hard-coded `4.0` once survived a
ladder change in nine places and would have silently reintroduced a 29×-too-strong
setting on any default run. `src/config.py` raises with an explanation rather than
guessing.

Edit the YAML by hand, or set `CHOSEN` below and run the cell. Either way, record
*why* in `operating_strength_source` — the G2b report path and the row of the
table you chose from.

In [ ]:
CHOSEN = None          # <-- e.g. 0.09, after reading the G2b report above
REASON = "chosen at G2b from artifacts/g2b/g2b_report.txt: <which row, and why>"

from pathlib import Path
import re, sys
sys.path.insert(0, "src")

if CHOSEN is None:
    import config as cfg_mod
    cfg = cfg_mod.load(Path.cwd())
    print("operating_strength is currently:",
          cfg["planned"].get("operating_strength"))
    print("Set CHOSEN above and re-run, or edit configs/sprint.yaml by hand.")
    print("\nFor reference, the ladder G2b swept:")
    print(" ", cfg_mod.strengths(cfg, include_extension=True))
else:
    path = Path("configs/sprint.yaml")
    text = path.read_text()
    text, n1 = re.subn(r"^(  operating_strength:).*$",
                       rf"\g<1> {CHOSEN}", text, count=1, flags=re.M)
    text, n2 = re.subn(r"^(  operating_strength_source:).*$",
                       rf'\g<1> "{REASON}"', text, count=1, flags=re.M)
    if n1 != 1:
        raise SystemExit("could not find planned.operating_strength in the yaml")
    path.write_text(text)
    import config as cfg_mod
    cfg = cfg_mod.load(Path.cwd())
    print("operating_strength =", cfg_mod.operating_strength(cfg))
    print("source             =", cfg["planned"]["operating_strength_source"])

## 9 — Concept vectors: band re-pilot, selection, both arms

`scripts/01_concept_vectors.py` filters the 500-concept pool to single-token
names, measures per-concept detection over the **whole** surviving pool **under
the band at the operating point**, selects 60 spanning the range, and extracts
both arms at **every** band layer.

**Why it is re-piloted.** The previous pilot scored 315 concepts at one layer,
where the injection was ~20× too weak: every rate sat on floor noise, so the 60
it selected were effectively random. Reselecting is *exploratory* and is recorded
as such in `configs/prereg.json` — it is a second pass at a selection step whose
first pass is known to be uninformative. If you are short of time you may keep
the existing 60 instead, and state that the selection was effectively random;
that is a defensible sprint choice, but it has to be said out loud.

**Why per-layer extraction.** Injecting a layer-27 vector at layer 38 is a
cross-layer mismatch, and under a 17-layer band 16 of the 17 injections would be
that mismatch. It costs nothing to fix: the 17 layers come off the same forward
pass the single-layer version already paid for, so 100 baselines + n concepts is
100 + n passes whether the band is 1 layer wide or 17.

**Why it measures its own rates at all.** The build spec stratifies on Macar's
per-concept Gemma detection rates. Those are not published — the metrics caches
are aggregates over `(layer_idx, strength, arm)`, the abliterated checkpoint is
weights only, and the README reports aggregates. Nothing reachable carries a
per-concept number. Consequences, all reported in G3: no cross-model transfer, so
no regression to the mean from another model's noisy labels; bimodality testable
on the full pool; but the cross-model Spearman cannot be computed at all, and
tier membership is not independent of detection, so **tiers are a sampling device
and no per-tier rate is reported anywhere**.

The pilot also gets the 10-way task round-robin, for the same reason the sweep
does: selecting 60 concepts on their response to one fixed prompt would make the
selection a property of that prompt.

In [ ]:
!python scripts/01_concept_vectors.py --per-tier 20 2>&1 | tail -40

In [ ]:
# what the re-pilot found, before G3 uses it
import json, sys
from pathlib import Path
import numpy as np
sys.path.insert(0, "src")
import stats

if not Path("artifacts/vectors/pilot.json").exists():
    print("artifacts/vectors/pilot.json not found -- run scripts/01_concept_vectors.py")
else:
    pilot = json.load(open("artifacts/vectors/pilot.json"))
    rates = np.array(list(pilot["rates"].values()))
    sel   = json.load(open("artifacts/vectors/selection.json"))
    band  = pilot.get("band_layers", [])
    print(f"pilot scored {rates.size} single-token concepts under "
          f"{pilot.get('injection_policy')} over layers "
          f"{band[0] if band else '?'}..{band[-1] if band else '?'}, "
          f"alpha_rel {pilot['strength']} per layer, "
          f"{pilot.get('n_tasks', 1)} tasks")
    print(f"  mean {rates.mean():.4f}   median {np.median(rates):.4f}")
    print(f"  >=0.9 : {(rates>=0.9).sum():>4}    <=0.01 : {(rates<=0.01).sum():>4}")
    print("  (run 2 put ~every concept at ~1e-3 here. A distribution that still")
    print("   has no spread means the band did not restore signal either, and")
    print("   the tiers are again a sampling device over noise.)")
    d = stats.dip_test(rates, n_boot=5000)
    print(f"  Hartigan dip {d['dip']:.4f}  p={d['p']:.4f}   <- bimodality on the FULL pool")
    print(f"  selected {len(sel['selected'])} concepts, "
          f"low tier via {sel['stratification'].get('low_tier_mode','?')}")
    print(f"  arms written {sel.get('arms')}"
          + (f"   jlens_row error: {sel['jlens_row_error']}"
             if sel.get("jlens_row_error") else ""))
    a = sel["composition_audit"]
    print(f"\nsingle-token filter: {a['n_before']} -> {a['n_after']}")
    for b, r in a["bucket_retention"].items():
        print(f"  {b:<9} {a['bucket_before'][b]:>4} -> {a['bucket_after'].get(b,0):<4} retention {r:.3f}")

## 10 — GATE G3: concept vectors and baseline reproduction

Now under the band, with the **(arm, strength)** grid — the layer axis is gone,
because there is no single injection layer any more. Every G3 number from the
single-layer run is **superseded, not comparable**: the per-layer α was the same
and the cumulative displacement differed by roughly the width of the band.

**What to read.** The **composition audit** first: if abstract concepts survive
the single-token filter at a different rate than concrete ones, any
abstract/concrete difference in detection is confounded before the experiment
starts. (The repo's own audit found abstract words survive *more* often here —
0.777 vs 0.578 — so the usual worry runs backwards on this tokenizer, and that
is worth saying in the paper.)

Then **TPR − FPR** and the **AUC column** — AUC is threshold-free and better
powered. Then the **identification rate split by arm**: `jlens_row` is the
ceiling, because it is Garcia's own injected object. A rate near zero *there* is
a harness fault, not a finding about the model, and it is the same stop
condition G2b sets. `concept` below `jlens_row` **at matched ‖δ‖** is the
finding — that is the verbalizability factor.

Then the **yes-bias arm**: if injecting a concept raises P(yes) on unrelated
questions whose truthful answer is "no", part of the detection signal is a
generic yes-bias from perturbing the residual stream, not introspection.

**What you will not see:** any per-tier number. Concepts were selected on
measured detection, so per-tier rates would regress to the mean by construction
— the spec's own table warns about exactly this. Tiers are a sampling device.

In [ ]:
!python gates/g3_baseline.py 2>&1 | tee artifacts/g3_console.log

### Optional: judge calibration

G3 writes 50 sampled responses to `artifacts/g3/to_label.json`. Label each
`1`/`0` for *"does this response name the injected concept?"*, save as
`artifacts/g3/hand_labels.json` in the form `{"<index>": 1, ...}`, and re-run G3
to get Cohen's kappa and the confusion matrix against the deterministic scorer.

Until then G3 prints kappa as PENDING rather than guessing at it. Set
`ANTHROPIC_API_KEY` to make the vendored LLM-judge rubrics available; without a
key the judge reports itself unavailable and never silently degrades to string
matching.

In [ ]:
import json
from pathlib import Path
path = Path("artifacts/g3/to_label.json")
if not path.exists():
    print("no to_label.json yet -- run gates/g3_baseline.py")
else:
    todo = json.load(open(path))
    print(f"{len(todo)} responses awaiting hand labels. First three:\n")
    for k, v in list(todo.items())[:3]:
        print(f"[{k}] concept={v['concept']!r}\n     {v['response'][:160]!r}\n")

## 11 — The sweep

**1,920 cells**: 60 concepts x 4 strengths x 2 orders x (2 arms + zero + random).
The layer dimension is gone — under the band there is one intervention at every
layer 24–40 at once, so "which layer" is no longer a coordinate. One shard per
concept, written atomically; re-running skips what exists.

**The controls are shared between the arms.** `control_zero` is exact to share:
α=0 adds exactly zero whatever the vector is. `control_random` is shared on
purpose — one null for both arms is what makes the A-vs-B contrast a contrast of
the injected *object* rather than of two different nulls.

**Ten tasks, not one.** Concept *i* takes `TASK_PROMPTS[i % 10]` in all of its
cells. Under one fixed task every `control_zero` cell was the same forward pass,
so FPR had no variance and the entire result was hostage to "name a colour of the
sky" — G3's own report flags this. The task is held fixed *within* a concept so
its injected and control cells stay paired: the contrast is the injection, not
the task. A batch is therefore a task group (6 concepts at 60/10), because only
concepts sharing a prompt can share a forward.

**Logits-first.** Each cell caches the residual at the report position for the
readout layers, the probe layer and the final block, plus the JSON-boolean
probabilities and the top-k logits. Full-vocabulary logits are never stored —
they are recoverable exactly from the cached final-layer residual, since unembed
is deterministic. The 17 band layers are *not* cached: nothing downstream reads a
residual at layer 31 of a 17-layer band, and caching them would quadruple the
shards to record the injection we already know we made.

**The report position had to be constructed.** "Next-token logits at the report
position" is unambiguous for report-first, where the model's first output token
*is* the detection verdict. For task-first the report lands inside the
generation, so reading both at the last prompt token would compare a detection
verdict against a task answer and call the difference an order effect. The
assistant turn is therefore prefilled up to the detection key in whichever order
the protocol demands, so the next token is a JSON boolean in both arms — and the
task-first prefill quotes the model's own greedy answer to *that concept's* task.

**The fingerprint.** A shard records the policy, band, norm_mode, arms, strengths
and task list it was produced under. Resuming on filename alone would silently
mix run 2's single-layer shards with these; the names are identical and the
numbers are ~20× apart.

In [ ]:
!python scripts/02_sweep.py --batch 8 2>&1 | tail -25

## 12 — Generation at the operating point

The expensive channel, so it runs at one strength for both orders, both arms and
the two shared controls: 60 x 2 x 4 x 4 samples = **1,920 generations**. (The
change order budgeted 2,880 by counting the controls once per arm; they are
shared here for the same reason they are shared in the sweep.)

**T=1.0 with 4 samples** — Garcia used T=0 with a single completion and flags it
as a limitation, so sampling with several draws is a cheap, legible improvement
over the closest prior work, and it is what makes f₃ an estimate with a spread
rather than one draw.

The run asserts that **every band layer fired exactly once** per generate call. A
hook that fired on a cached decode step would be editing the model's own output
as it wrote it, which would look exactly like introspection.

Resumable per task group. This is the long pole; if the runtime drops, re-run the
same cell and it picks up where it stopped.

In [ ]:
!python scripts/03_generate.py --samples 4 2>&1 | tail -20

## 13 — GATE G4a: sweep integrity

Audits the artifacts, not the model — **no GPU needed**, so you can re-run it any
time to see what actually landed.

**What it is for:** the failures that leave no error message. A missing cell. A
shard written twice under the wrong name (caught by hashing residual payloads —
two concepts with byte-identical residuals across every cell is not a
coincidence). A NaN that propagated. And **temporal drift**: the zero-strength
control compared between the first and last decile of shards by write time. A
systematic shift there means something changed mid-run — a reloaded model, a
different dtype, a hardware switch — and every number in the run inherits it.

**New under the band: the band-integrity block.** Fires per injected cell must be
17 (one per layer, prefill only) and 0 per zero cell; realised ‖δ_ℓ‖ / base_ℓ must
equal the α the cell claims, exactly, because the hook unit-normalises before
scaling. This is the check that a cell labelled α=0.09 was actually run at 0.09 —
the failure that produced two invalid runs was precisely a label that no longer
described the intervention.

**Also new: zero-strength detection by task.** That table is the variance the FPR
interval rests on. A flat column means the 10-way assignment is not reaching the
control channel.

Also read `P(true)+P(false)`. If that sum is not near 1, the prefill is not
landing where it was meant to and the next token is not a JSON boolean at all,
which would quietly invalidate the whole detection channel.

In [ ]:
!python gates/g4a_sweep.py 2>&1 | tee artifacts/g4a_console.log

## 14 — Factor inputs, once per arm (the GPU stage)

`scripts/04_factors.py` turns sweep shards and generations into the arrays the
cascade needs: lens readouts from the cached residuals, and the **position
control**. It runs **once per arm** and writes `artifacts/factors/<arm>/`.

**Why one arm per run.** A cascade is a cascade *of something*: f₁, f₂ and f₃ all
have to be measured on the same injected object for the product to telescope. The
sweep carries both arms in one grid; this stage selects one arm's `injected` cells,
pairs them with the shared controls, and hands G4 something it can read without
knowing arms exist.

**The position control was wrong and is fixed.** It drew its comparison position
from `[0, first_injected)` — the system-prompt prefix, the same tokens in every
trial and causally *upstream* of the injection, so it could not have carried the
concept even in principle. It now draws from all non-injected, non-report
positions, which includes the tokens *after* the span where leakage would
actually show up.

**Why the generator continues from the sweep's own prompt.** f₁ and f₂ are
properties of a prefill residual; f₃ is a property of the continuation sampled
from it. If the two came from different prompts, "the same trial" would be
undefined and `cascade_residual` would measure that inconsistency instead of a
denominator bug.

**f₁ is probed at layer 40** — the band's last layer, the deepest point the
intervention is still being written to, and the report position's own depth.
Probing at 27 under a 24–40 band would ask whether the first quarter of the
intervention is detectable, not whether the intervention is.

In [ ]:
import sys, subprocess
sys.path.insert(0, "src")
import config as cfg_mod
from pathlib import Path

for arm in cfg_mod.vector_arms(cfg_mod.load(Path.cwd())):
    print("=" * 70)
    print("04_factors, arm:", arm)
    print("=" * 70)
    subprocess.run([sys.executable, "scripts/04_factors.py",
                    "--vector-arm", arm], check=False)

## 15 — GATE G4: the three-factor decomposition, per arm

    P(report) = P(represented) x P(verbalizable | represented) x P(reported | verbalizable)

No GPU. **Read `cascade_residual` first** — it should be floating-point zero at
every k and in both orders. The chain rule makes the product exact when the
denominators nest, so anything above ~1e-12 means a stage dropped trials or took
a different denominator, and every number below it is suspect. It is `nan`, not
zero, wherever nothing survives to f₃ — that is undefined, not a failure.

Then **survivorship**. If only a handful of trials reach f₃, the f₃ number is
fragile no matter how tight its interval looks; the jackknife range and the
judge-noise interval bound how much to trust it.

Then the **controls**: the probe null (shuffled labels) should collapse to the
FPR the threshold was set at — 0.05, not 0. The **f₂ null band** matters because
cosine has no absolute scale at d=5120, so f₂ is only meaningful against the
matched-norm random arm.

**Reading the two arms together.** `jlens_row` is the ceiling — Garcia's own
object, ~pure J-space. `concept` is the headline — the same norm, but only ~6–7%
of its variance in J-space (G2b measures this as cos² between the two). If B
survives f₂ and f₃ where A does not, at matched ‖δ‖, that is the verbalizability
factor demonstrated causally on open weights rather than argued for.

In [ ]:
import sys, subprocess
sys.path.insert(0, "src")
import config as cfg_mod
from pathlib import Path

for arm in cfg_mod.vector_arms(cfg_mod.load(Path.cwd())):
    print("=" * 70)
    print("G4, arm:", arm)
    print("=" * 70)
    subprocess.run([sys.executable, "gates/g4_factors.py",
                    "--arm", arm, "--k", "10"], check=False)

In [ ]:
# the cascade at a glance, both arms side by side, straight from the artifacts
import json, sys
from pathlib import Path
sys.path.insert(0, "src")
import config as cfg_mod

arms = cfg_mod.vector_arms(cfg_mod.load(Path.cwd()))
rows = {}
for arm in arms:
    path = Path("artifacts/g4") / arm / "g4_factors.json"
    if path.exists():
        rows[arm] = json.load(open(path))

if not rows:
    print("no g4_factors.json yet. Run, in order:")
    print("  g2b_band -> (choose the operating point) -> 01_concept_vectors")
    print("  -> 02_sweep -> 03_generate -> 04_factors -> g4_factors")
else:
    print(f"{'':<34}" + "".join(f"{a:>16}" for a in rows))
    def line(label, fn):
        print(f"{label:<34}" + "".join(f"{fn(g):>16}" for g in rows.values()))
    line("f1 (represented)", lambda g: f"{g['headline']['f1']:.4f}")
    line("f2 (verbalizable | represented)", lambda g: f"{g['headline']['f2']:.4f}")
    line("f3 (reported | verbalizable)", lambda g: f"{g['headline']['f3']:.4f}")
    line("product", lambda g: f"{g['headline']['f1']*g['headline']['f2']*g['headline']['f3']:.6f}")
    line("observed", lambda g: f"{g['headline']['observed_cascade_rate']:.6f}")
    line("cascade_residual  <- must be ~0", lambda g: f"{g['headline']['residual']:.3e}")
    line("survivorship", lambda g: "{}->{}->{}->{}".format(
        g['headline']['n_entering_f1'], g['headline']['n_surviving_f1'],
        g['headline']['n_surviving_f2'], g['headline']['n_surviving_f3']))
    line("naive report rate", lambda g: f"{g['naive_report_rate']:.4f}")
    print()
    print("losses: representation failure, verbalizability failure, channel closure")
    line("  1 - f1", lambda g: f"{1-g['headline']['f1']:.4f}")
    line("  1 - f2", lambda g: f"{1-g['headline']['f2']:.4f}")
    line("  1 - f3", lambda g: f"{1-g['headline']['f3']:.4f}")

## 16 — Figures and GATE G5, per arm

No GPU. `05_figures.py` recomputes the cascade from `factors_input.npz` rather
than reading the G4 report, so the figures stand alone — a judge can regenerate
them without having run any gate, which is what the demo notebook does. It also
writes `RESULTS.md`, the numbers block the README quotes, generated rather than
typed so the two cannot drift apart.

G5 collects every control arm into one table and asks the four questions that
decide whether a null is a finding: is it FPR-matched, is it multiplicity-
corrected (BH q-values on the headline claims), is it powered (minimum detectable
effect at 80%), and was it pre-registered. `configs/prereg.json` now records the
band policy, the two arms, the task round-robin, the reselected 60 and the
position-control fix — in both directions, specified and exploratory.

In [ ]:
import sys, subprocess
sys.path.insert(0, "src")
import config as cfg_mod
from pathlib import Path

for arm in cfg_mod.vector_arms(cfg_mod.load(Path.cwd())):
    print("=" * 70)
    print("figures + G5, arm:", arm)
    print("=" * 70)
    subprocess.run([sys.executable, "scripts/05_figures.py",
                    "--arm", arm, "--k", "10"], check=False)
    subprocess.run([sys.executable, "gates/g5_controls.py",
                    "--arm", arm], check=False)

## 17 — Collect the reports

Everything is on disk. Download `artifacts/` before the runtime recycles.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, "src")
import config as cfg_mod

for p in sorted(Path("artifacts").rglob("*")):
    if p.is_file() and p.suffix != ".npz":
        print(f"{p.stat().st_size:>12,}  {p}")
shards = list(Path("artifacts/sweep").glob("shard_*.npz")) if Path("artifacts/sweep").exists() else []
print(f"\n{len(shards)} sweep shards, {sum(p.stat().st_size for p in shards)/2**20:.1f} MiB")

arms = cfg_mod.vector_arms(cfg_mod.load(Path.cwd()))
print("\n" + "="*70)
print("re-read any report without re-running its gate:")
for g in ("g0","g1","g2","g2b","g3","g4a"):
    print(f"  print(open('artifacts/{g}/{g}_report.txt').read())")
for arm in arms:
    print(f"  print(open('artifacts/g4/{arm}/g4_report.txt').read())")
    print(f"  print(open('artifacts/g5/{arm}/g5_report.txt').read())")

In [ ]:
# zip the artifacts for download
!zip -qr artifacts.zip artifacts && ls -lh artifacts.zip
try:
    from google.colab import files
    files.download("artifacts.zip")
except Exception as e:
    print("not on Colab, download artifacts.zip manually:", type(e).__name__)